# Emotions classification from Images

- Now you can download the fer2013_2_classes.zip file which is in the Datasets folder in the Google Drive. This new dataset is composed of only angry and happy images, so you can build a classifier in order to detect only those emotions and compare the results

- Try to play with the parameters and also with the neural network structure to see how the results change. You can also try to use the original fer2013 dataset and build a classifier for all the emotions.

### Import libraries and modules

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns   
import zipfile
import tensorflow as tf

from google.colab.patches import cv2_imshow
from google.colab import drive

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv2D, MaxPooling2D, Flatten, BatchNormalization

from keras.models import save_model

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

### Loading the images

In [ ]:
drive.mount('/content/drive')
# Define paths for images, classifiers and models
image_folder_path = '/content/drive/My Drive/Colab Notebooks/Computer Vision Masterclass_Udemy/Training_Material/'

In [ ]:
path = image_folder_path +'/Datasets/fer2013_2_classes.zip'
zip_object = zipfile.ZipFile(file=path, mode='r')
zip_object.extractall('./')
zip_object.close()

In [ ]:
tf.keras.preprocessing.image.load_img('/content/fer2013_2_classes/train/Angry/1003.jpg')

In [ ]:
image = tf.keras.preprocessing.image.load_img('/content/fer2013_2_classes/train/Happy/1.jpg')
image

### Train and Test dataset

In [ ]:
# Create an instance of the ImageDataGenerator class for data augmentation, which will generate batches of tensor image data with real-time data augmentation. 
# The data will be looped over (in batches) indefinitely. The ImageDataGenerator class allows you to configure random transformations and normalization operations to be done on your image data during training. This is useful for training deep learning models as it helps to prevent overfitting and improve generalization. 
# Also, it can be used to rescale pixel values, apply random transformations like rotation, flipping, shifting, and zooming to the images in the training dataset. This helps to create a more diverse set of training examples and improve the model's ability to generalize to new data. 
training_generator = ImageDataGenerator(
    rescale=1./255,         # Rescale the pixel values 
    rotation_range=7,       # Randomly rotate images in the range (degrees, 0 to 180)
    horizontal_flip=True,   # Randomly flip inputs horizontally
    vertical_flip=True,    # Randomly flip inputs vertically
    width_shift_range=0.2,  # Randomly translate images horizontally (fraction of total width)
    zoom_range=0.2          # Randomly zoom images
)

# Train dataset is created using the flow_from_directory method of the ImageDataGenerator class. This method takes the path to the directory containing the training images, along with other parameters such as target size, batch size, class mode, and shuffle. 
# The target size specifies the dimensions to which all images will be resized, while the batch size determines how many images will be processed in each batch. The class mode specifies how the labels are generated (in this case, categorical), and shuffle indicates whether to shuffle the data after each epoch.
train_dataset = training_generator.flow_from_directory(
    '/content/fer2013_2_classes/train',   # Path to the directory containing the training images
    target_size = (48, 48),     # Resize all images to 48x48 pixels
    batch_size = 16,            # Number of images to be yielded from the generator per batch
    class_mode = 'categorical', # Specify the type of label arrays that are returned: 'categorical' means 2D one-hot encoded labels
    shuffle = True,             # Whether to shuffle the data after each epoch
)

In [ ]:
train_dataset.classes

In [ ]:
# Get the unique classes and their counts in the training dataset using numpy's unique function. This will return an array of unique class labels and an array of their corresponding counts.
np.unique(train_dataset.classes, return_counts=True)

In [ ]:
# Get the class indices mapping from the training dataset. This will return a dictionary where the keys are the class labels and the values are the corresponding integer indices assigned to each class.
train_dataset.class_indices

In [ ]:
sns.countplot(x = train_dataset.classes);

In [ ]:
test_generator = ImageDataGenerator(rescale=1./255)

test_dataset = test_generator.flow_from_directory(
    '/content/fer2013_2_classes/validation', 
    target_size = (48, 48), 
    batch_size = 1, 
    class_mode = 'categorical', 
    shuffle = False
)

### Building and training the convolutional neural network

In [ ]:
num_detectors = 32
num_classes = 2
width, height = 48, 48
epochs = 50

network = Sequential()

network.add(Conv2D(num_detectors, (3, 3), activation = 'relu', padding = 'same', input_shape = (width, height, 3)))
network.add(BatchNormalization())
network.add(Conv2D(num_detectors, (3, 3), activation = 'relu', padding = "same"))
network.add(BatchNormalization())
network.add(MaxPooling2D(pool_size=(2, 2)))
network.add(Dropout(0.2))

network.add(Conv2D(2*num_detectors, (3, 3), activation = 'relu', padding="same"))
network.add(BatchNormalization())
network.add(Conv2D(2*num_detectors, (3, 3), activation = 'relu', padding="same"))
network.add(BatchNormalization())
network.add(MaxPooling2D(pool_size=(2, 2)))
network.add(Dropout(0.2))

network.add(Conv2D(2*2*num_detectors, (3, 3), activation = 'relu', padding="same"))
network.add(BatchNormalization())
network.add(Conv2D(2*2*num_detectors, (3, 3), activation = 'relu', padding="same"))
network.add(BatchNormalization())
network.add(MaxPooling2D(pool_size=(2, 2)))
network.add(Dropout(0.2))

network.add(Conv2D(2*2*2*num_detectors, (3, 3), activation = 'relu', padding="same"))
network.add(BatchNormalization())
network.add(Conv2D(2*2*2*num_detectors, (3, 3), activation = 'relu', padding="same"))
network.add(BatchNormalization())
network.add(MaxPooling2D(pool_size=(2, 2)))
network.add(Dropout(0.2))

network.add(Flatten())
network.add(Dense(2*num_detectors, activation = 'relu'))
network.add(BatchNormalization())
network.add(Dropout(0.2))

network.add(Dense(2*num_detectors, activation = 'relu'))
network.add(BatchNormalization())
network.add(Dropout(0.2))

network.add(Dense(num_classes, activation='softmax'))

print(network.summary())

In [ ]:
network.compile(loss='categorical_crossentropy', optimizer='Adam', metrics=['accuracy'])

In [ ]:
network.fit(train_dataset, epochs=epochs)

### Evaluate neural network

In [ ]:
# Evaluate the loaded model on the test dataset to assess its performance. The evaluate method computes the loss and accuracy metrics for the model on the provided test dataset, allowing you to see how well the model generalizes to unseen data. 
network.evaluate(test_dataset)

In [ ]:
# Generate predictions for the test dataset using the loaded model. The predict method takes the test dataset as input and returns the predicted probabilities for each class for each image in the test dataset. These predictions can then be used to evaluate the model's performance or to make decisions based on the predicted classes.
predictions = network.predict(test_dataset)
predictions

In [ ]:
# Convert the predicted probabilities into class labels by taking the index of the maximum value along the second axis (axis=1) for each prediction. This will give you the predicted class label for each image in the test dataset, which can then be compared to the true labels to evaluate the model's performance.
predictions = np.argmax(predictions, axis = 1)
predictions

In [ ]:
test_dataset.classes

In [ ]:
accuracy_score(test_dataset.classes, predictions)

In [ ]:
test_dataset.class_indices

In [ ]:
cm = confusion_matrix(test_dataset.classes, predictions)
cm

In [ ]:
sns.heatmap(cm, annot=True);

In [ ]:
# Generate a classification report for the test dataset using the true labels and predicted labels. The classification report provides a summary of the precision, recall, F1-score, and support for each class in the dataset. This information can be used to evaluate the performance of the model on each emotion class and identify any areas where the model may need improvement.
print(classification_report(test_dataset.classes, predictions))